# 9. Integrita dat, bezpečnost, logování, kontrola vstupů, zpracování chyb

### Integrita dat
* Zajišťuje správnost, úplnost a konzistenci dat v programu
* Přispívá k ní architektonický princip SSOT (Single Source of Truth), každý zdroj dat je načítán a ukládán z jednoho místa v programu

### Bezpečnost
* Nevalidovaný vstup může způsobit SQL injection
* Prevencí je ošetření vstupů, využívání návrhových vzorů jako DAO a využívání Prepared statements
* Součástí bezpečnosti je i ochrana citlivých dat, nesmí se ukládat v čistém textu, převádí se na nečitelné řetězce pomocí hashovacích algoritmů

### Logování
* Průběžný záznam událostí v aplikaci
* Log lze použít k rekonstrukci chyby
* Vývojář přesně ví, v jakém stavu se program nacházel před svým pádem

### Kontrola vstupů
* Načíst data od uživatele vždy vyžaduje kontrolu vstupů
* Musí obsahovat validaci datových typů, povolených rozsahů a implementaci cyklů pro opakované zadání
* Nástrojem pro kontrolu přesného formátu dat jsou regulární výrazy a funkce `re.match()`

### Zpracování chyb
* Kriticky důležité chybovým stavům předcházet a zachytit je
* Nezachycená výjimka ukončí proces i vlákno
* Využívají se k tomu bloky `try/catch` (v Pythonu `try/except`), můžeme v nic vzniklou výjimku zpracovat a aplikaci zachránit před pádem

In [ ]:
import re
import logging
import hashlib

**1. LOGOVÁNÍ, KONTROLA VSTUPŮ, BEZPEČNOST A INTEGRITA**

In [ ]:
# Nastavení formátu logování (zprávy se obvykle zapisují do souboru, zde do konzole)
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

class BankovniUcet:
    def __init__(self, cislo_uctu: str, pin: str):
        if not re.match(r"^[0-9]{5,10}/[0-9]{4}$", cislo_uctu):
            logging.error(f"Uživatel zadal neplatný formát účtu: {cislo_uctu}")
            # Vynucení chybového stavu při špatném vstupu
            raise ValueError('Číslo účtu neodpovídá formátu zapisu 000000000/0000.')

        self._cislo_uctu = cislo_uctu

        # Nikdy neukládáme PIN v čistém textu, vytvoříme jeho otisk algoritmem SHA-384
        self._pin_hash = hashlib.sha384(pin.encode('utf-8')).hexdigest()
        self._zustatek = 0.0

        logging.info(f"Účet chráněn a úspěšně vytvořen (hash: {self._pin_hash[:10]}...).")

**4. ZPRACOVÁNÍ CHYB**

In [ ]:
def bezpecne_zalozeni_uctu(ucet_str, pin_str):
    try:
        # Pokus o riskantní operaci
        novy_ucet = BankovniUcet(ucet_str, pin_str)
        return novy_ucet
    except Exception as e:
        # Nezachycená výjimka by ukončila proces, takto program bezpečně pokračuje
        logging.warning(f"Zpracována chyba při zakládání účtu: {e}")
        return None

print("=== START APLIKACE ===")

# Validní vstup (projde bez chyb a zapíše info log)
ucet_ok = bezpecne_zalozeni_uctu("123456789/0100", "1234")

# Nevalidní vstup (narazí na regex kontrolu, zaloguje error a varování,
# ale program díky try/except nespadne!)
ucet_fail = bezpecne_zalozeni_uctu("NespravnyFormatUctu", "9999")

print("=== KONEC APLIKACE (Program nespadl) ===")